In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import json

# Preprocessing Offshore dataset

In [38]:
path = "../datasets_raw/Analyzing_Europes_Biggest_Offshore_Wind_Farms/"
raw_filenames = ["Borssele_(Phase_1,2).csv", "Borssele_(Phase_3,4).csv","Gemini.csv","Hollandse_Kust_Noord.csv","Hollandse_Kust_Zuid.csv"]
wind_farm_names = ["Borssele_12","Borssele_34","Gemini", "Hollandse_Kust_Noord", "Hollandse_Kust_Zuid"]

dfs = []

for i, file in enumerate(raw_filenames):
    df = pd.read_csv(path + file)
    #drop unused columns
    df = df.drop(columns=["u100", "v100","fsr", "Unnamed: 0"])
    df = df.drop(columns=[col for col in df.columns if col.startswith("Power_of_")])
    # Rename Scaled Windspeed
    df.columns = [col if not col.startswith("Scaled_Windspeed") else "Scaled_Windspeed" for col in df.columns]
    df = df.set_index("time")
    df.index = pd.to_datetime(df.index)
    df["Station"] = wind_farm_names[i]
    dfs.append(df)

### Quality Checks

In [39]:
with open("../datasets/metadata_wind_farms.json") as f:
    wind_farms_metadata = json.load(f)["wind_farms"]

capacities = {farm["id"]: farm["installed_capacity_mw"] for farm in wind_farms_metadata}

for df, name in zip(dfs, wind_farm_names):
    capacity = capacities[name]
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")

    # ── Duplicate timestamps
    dupes = df[df.index.duplicated()]
    print(f"Duplicate timestamps:       {len(dupes)}")

    # ── Missing timestamps
    expected = pd.date_range(start=df.index.min(), end=df.index.max(), freq="h")
    missing = expected.difference(df.index)
    print(f"Missing timestamps:         {len(missing)}")

    # ── Missing values
    print(f"Missing values:\n{df.isna().sum()}")

    # ── Wind direction out of range
    print(f"Wind direction < 0:         {(df['Wind_Direction'] < 0).sum()}")
    print(f"Wind direction > 360:       {(df['Wind_Direction'] > 360).sum()}")

    # ── Physically impossible values
    print(f"Negative wind speed:        {(df['Scaled_Windspeed'] < 0).sum()}")
    print(f"Wind speed > 50 m/s:        {(df['Scaled_Windspeed'] > 50).sum()}")
    print(f"Negative power:             {(df['Power'] < 0).sum()}")

    # ── Cap power at installed capacity
    over_capacity = (df["Power"] > capacity).sum()
    print(f"Power > capacity:           {over_capacity}")
    if over_capacity > 0:
        df.loc[df["Power"] > capacity, "Power"] = capacity
        print(f"  -> Capped {over_capacity} values to {capacity} MW")
        print(f"Power > capacity fixed:           {(df['Power'] > capacity).sum()}")

    # ── Turn_off inconsistencies
    flag_on_no_power = (df["Turn_off"] == 0) & (df["Power"] > 0)
    flag_off_has_power = (df["Turn_off"] == 1) & (df["Power"] == 0)
    print(f"Turn_off=0 but Power > 0:   {flag_on_no_power.sum()}")
    print(f"Turn_off=1 but Power = 0:   {flag_off_has_power.sum()}")

    df.to_csv("../datasets/offshore_"+ wind_farm_names[i] + ".csv")


  Borssele_12
Duplicate timestamps:       0
Missing timestamps:         0
Missing values:
Windspeed           0
Scaled_Windspeed    0
Wind_Direction      0
Turn_off            0
Power               0
Station             0
dtype: int64
Wind direction < 0:         0
Wind direction > 360:       0
Negative wind speed:        0
Wind speed > 50 m/s:        0
Negative power:             0
Power > capacity:           0
Turn_off=0 but Power > 0:   0
Turn_off=1 but Power = 0:   23224

  Borssele_34
Duplicate timestamps:       0
Missing timestamps:         0
Missing values:
Windspeed           0
Scaled_Windspeed    0
Wind_Direction      0
Turn_off            0
Power               0
Station             0
dtype: int64
Wind direction < 0:         0
Wind direction > 360:       0
Negative wind speed:        0
Wind speed > 50 m/s:        0
Negative power:             0
Power > capacity:           0
Turn_off=0 but Power > 0:   0
Turn_off=1 but Power = 0:   23760

  Gemini
Duplicate timestamps:       0
